In [1]:
import os
import pandas as pd
import pydicom
from tqdm import tqdm

ROOT = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI"

rows = []

for root, _, files in os.walk(ROOT):
    for f in files:
        if f.endswith(".dcm"):
            path = os.path.join(root,f)

            try:
                d = pydicom.dcmread(path, stop_before_pixels=True)

                rows.append({
                    "PatientID": str(d.PatientID),
                    "StudyUID": str(d.StudyInstanceUID),
                    "SeriesUID": str(d.SeriesInstanceUID),
                    "SeriesDescription": str(getattr(d,"SeriesDescription","")),
                    "FilePath": path
                })

            except:
                pass

df = pd.DataFrame(rows)

df.to_csv("dicom_index.csv",index=False)

print("Total studies:",df["StudyUID"].nunique())
print("Total series:",df["SeriesUID"].nunique())    

c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '2.16.124.113543.6006.99.09016870996506928903'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '2.16.124.113543.6006.99.03087560979650955183'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '2.16.124.113543.6006.99.08346806387959279304'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\Acer

Total studies: 451
Total series: 831


In [2]:
import pandas as pd

df = pd.read_csv("dicom_index.csv")

DROP = ["B1-CALIBRATION","LOCALIZER","SCOUT"]
KEEP = ["MPRAGE","MP-RAGE","FSPGR","SPGR","IR-FSPGR"]

def contains(text,keywords):
    text = str(text).upper()
    return any(k in text for k in keywords)

df = df[~df["SeriesDescription"].apply(lambda x: contains(x,DROP))]
df = df[df["SeriesDescription"].apply(lambda x: contains(x,KEEP))]

print("Remaining series:",df["SeriesUID"].nunique())

Remaining series: 624


In [3]:
series = df.drop_duplicates("SeriesUID")

def score(desc):
    desc=str(desc).upper()

    s=0
    if "MPRAGE" in desc:
        s+=10
    if "REPEAT" in desc:
        s-=5

    return s

series["score"]=series["SeriesDescription"].apply(score)

best=(series
      .sort_values(["StudyUID","score"],ascending=[True,False])
      .drop_duplicates("StudyUID"))

best.to_csv("selected_scans.csv",index=False)

print("Final MRI volumes:",len(best))

Final MRI volumes: 449


C:\Users\Acer\AppData\Local\Temp\ipykernel_26292\406685196.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series["score"]=series["SeriesDescription"].apply(score)


In [8]:
import shutil

print(shutil.which("dcm2niix"))

C:\Users\Acer\AppData\Local\Programs\Python\Python310\Scripts\dcm2niix.EXE


In [9]:
import subprocess
import os
import pandas as pd
from tqdm import tqdm

DCM2NIIX=r"C:\Users\Acer\AppData\Local\Programs\Python\Python310\Scripts\dcm2niix.EXE"

OUT=r"nifti_output"

df=pd.read_csv("selected_scans.csv")

os.makedirs(OUT,exist_ok=True)

for _,row in tqdm(df.iterrows(),total=len(df)):

    dcm=row["FilePath"]
    folder=os.path.dirname(dcm)

    outfolder=os.path.join(OUT,row["PatientID"])
    os.makedirs(outfolder,exist_ok=True)

    subprocess.run([
        DCM2NIIX,
        "-z","y",
        "-o",outfolder,
        folder
    ])

100%|██████████| 449/449 [1:03:12<00:00,  8.45s/it]


In [11]:
import pyreadr

file_path = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\DXSUM.rda"

result = pyreadr.read_r(file_path)

print(result.keys())

odict_keys(['DXSUM'])


c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\pyreadr\_pyreadr_parser.py:233: RuntimeWarning: invalid value encountered in cast
  df[colname] = df[colname].values.astype("datetime64[D]").astype(datetime)


In [12]:
df = result["DXSUM"]

print(df.head())
print(df.columns)
print(df.shape)

  ORIGPROT COLPROT        PTID  RID VISCODE VISCODE2    EXAMDATE DIAGNOSIS  \
0    ADNI1   ADNI1  011_S_0002  2.0      bl       bl  2005-09-29        CN   
1    ADNI1   ADNI1  011_S_0003  3.0      bl       bl  2005-09-30  Dementia   
2    ADNI1   ADNI1  011_S_0005  5.0      bl       bl  2005-09-30        CN   
3    ADNI1   ADNI1  011_S_0008  8.0      bl       bl  2005-09-30        CN   
4    ADNI1   ADNI1  022_S_0007  7.0      bl       bl  2005-10-06  Dementia   

  DXNORM DXNODEP  ... DXODES              DXCONFID    ID SITEID    USERDATE  \
0    Yes     NaN  ...    NaN      Highly Confident   2.0  107.0  2005-10-01   
1    NaN     NaN  ...    NaN  Moderately Confident   4.0  107.0  2005-10-01   
2    Yes     NaN  ...    NaN      Highly Confident   6.0  107.0  2005-10-01   
3    Yes     NaN  ...    NaN  Moderately Confident   8.0  107.0  2005-10-01   
4    NaN     NaN  ...    NaN      Highly Confident  10.0   10.0  2005-10-06   

  USERDATE2 DD_CRF_VERSION_LABEL LANGUAGE_CODE HAS_QC_ER

In [14]:
clin = df[df["VISCODE"] == "bl"][["PTID","DIAGNOSIS"]]

clin = clin.rename(columns={
    "PTID":"PatientID",
    "DIAGNOSIS":"Label"
})

print(clin.head())
print(clin["Label"].value_counts())

    PatientID     Label
0  011_S_0002        CN
1  011_S_0003  Dementia
2  011_S_0005        CN
3  011_S_0008        CN
4  022_S_0007  Dementia
Label
MCI         767
CN          608
Dementia    266
Name: count, dtype: int64


In [15]:
clin["Label"] = clin["Label"].replace({
    "Dementia": "AD"
})

print(clin["Label"].value_counts())

Label
MCI    767
CN     608
AD     266
Name: count, dtype: int64


In [16]:
mri = pd.read_csv("selected_scans.csv")

data = mri.merge(clin, on="PatientID")

print("Total labeled MRI scans:", len(data))
print(data["Label"].value_counts())

Total labeled MRI scans: 394
Label
MCI    212
CN     120
AD      62
Name: count, dtype: int64


In [17]:
data.to_csv("mri_labeled_dataset.csv", index=False)

In [21]:
import os
import pandas as pd

NIFTI_ROOT = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output"

df = pd.read_csv("mri_labeled_dataset.csv")
df["PatientID"] = df["PatientID"].astype(str).str.strip().str.upper()

def find_nifti(pid):
    p = os.path.join(NIFTI_ROOT, pid)
    if not os.path.isdir(p):
        return None
    for root, _, files in os.walk(p):
        for f in files:
            if f.lower().endswith(".nii.gz") or f.lower().endswith(".nii"):
                return os.path.join(root, f)
    return None

df["NiftiPath"] = df["PatientID"].apply(find_nifti)
print("Labeled MRIs with NIfTI found:", df["NiftiPath"].notna().sum())

df2 = df.dropna(subset=["NiftiPath"]).copy()
df2.to_csv("mri_labeled_with_nifti.csv", index=False)
print("Saved mri_labeled_with_nifti.csv with rows:", len(df2))

Labeled MRIs with NIfTI found: 394
Saved mri_labeled_with_nifti.csv with rows: 394


In [ ]:
import os

SEARCH_ROOT = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA"  

nii_paths = []
for root, _, files in os.walk(SEARCH_ROOT):
    for f in files:
        if f.lower().endswith(".nii") or f.lower().endswith(".nii.gz"):
            nii_paths.append(os.path.join(root, f))

print("Found NIfTI files:", len(nii_paths))
print("First 10 paths:")
for p in nii_paths[:10]:
    print(p)

Found NIfTI files: 450
First 10 paths:
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\002_S_1155\I995496_Accelerated_Sagittal_MPRAGE_20180508130641_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\002_S_1261\I989320_Accelerated_Sagittal_MPRAGE_20180424082009_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\002_S_4799\I1010814_Accelerated_Sagittal_MPRAGE_20180614075810_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\002_S_6007\I988538_Accelerated_Sagittal_MPRAGE_20180418081255_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\002_S_6652\I1292937_Accelerated_Sagittal_MPRAGE_20200213121311_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\003_S_0908\I1082571_Accelerated_Sagittal_MPRAGE_20181205133149_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\003_S_1122\I1020423_Accelerated_Sagittal_MPRAGE_20180712112517_2.nii.gz
C:\Users\Acer\Downloads\XAI-ADNI-DATA\ADNI\nifti_output\003_S_1122\I

In [22]:
import os

def find_best_nifti(pid):
    p = os.path.join(NIFTI_ROOT, pid)
    if not os.path.isdir(p):
        return None

    candidates = []
    for root, _, files in os.walk(p):
        for f in files:
            if f.lower().endswith(".nii.gz") or f.lower().endswith(".nii"):
                fp = os.path.join(root, f)
                candidates.append((os.path.getsize(fp), fp))

    if not candidates:
        return None

    candidates.sort(reverse=True)  # largest first
    return candidates[0][1]

In [ ]:
import pandas as pd

df = pd.read_csv("mri_labeled_with_nifti.csv")  
print("Rows:", len(df))
print(df["Label"].value_counts())

Rows: 394
Label
MCI    212
CN     120
AD      62
Name: count, dtype: int64


In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("mri_labeled_with_nifti.csv")
df["PatientID"] = df["PatientID"].astype(str).str.strip().str.upper()

subjects = df["PatientID"].unique()

train_ids, temp_ids = train_test_split(subjects, test_size=0.30, random_state=42, shuffle=True)
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.50, random_state=42, shuffle=True)

def assign_split(pid):
    if pid in train_ids: return "train"
    if pid in val_ids: return "val"
    return "test"

df["split"] = df["PatientID"].apply(assign_split)

print(df["split"].value_counts())
print(df.groupby(["split","Label"]).size())

df.to_csv("mri_split.csv", index=False)
print("Saved: mri_split.csv")

split
train    275
val       60
test      59
Name: count, dtype: int64
split  Label
test   AD        11
       CN        22
       MCI       26
train  AD        38
       CN        78
       MCI      159
val    AD        13
       CN        20
       MCI       27
dtype: int64
Saved: mri_split.csv


In [25]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from PIL import Image

OUT_DATASET = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA\dataset_images"
N_SLICES = 40
TARGET_SIZE = (224, 224)

df = pd.read_csv("mri_split.csv")

def normalize_slice(x):
    x = x.astype(np.float32)
    p1, p99 = np.percentile(x, 1), np.percentile(x, 99)
    if (p99 - p1) < 1e-6:
        return None
    x = np.clip((x - p1) / (p99 - p1), 0, 1)
    return (x * 255).astype(np.uint8)

total_saved = 0
failed = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting slices"):
    nifti_path = row["NiftiPath"]
    split = row["split"]
    label = row["Label"]
    pid = row["PatientID"]

    try:
        vol = nib.load(nifti_path).get_fdata()
    except Exception:
        failed += 1
        continue

    # axial slices
    z = vol.shape[2]
    mid = z // 2
    half = N_SLICES // 2

    start = max(0, mid - half)
    end = min(z, start + N_SLICES)
    start = max(0, end - N_SLICES)

    out_dir = os.path.join(OUT_DATASET, split, label, str(pid))
    os.makedirs(out_dir, exist_ok=True)

    for zi in range(start, end):
        sl = vol[:, :, zi]
        sl = normalize_slice(sl)
        if sl is None:
            continue

        img = Image.fromarray(sl)
        img = img.resize(TARGET_SIZE)

        img.save(os.path.join(out_dir, f"{pid}_z{zi:03d}.png"))
        total_saved += 1

print("Total slice images saved:", total_saved)
print("Failed volumes:", failed)
print("Dataset created at:", OUT_DATASET)

Extracting slices: 100%|██████████| 394/394 [04:00<00:00,  1.63it/s]

Total slice images saved: 15760
Failed volumes: 0
Dataset created at: C:\Users\Acer\Downloads\XAI-ADNI-DATA\dataset_images


In [26]:
import os

root = r"C:\Users\Acer\Downloads\XAI-ADNI-DATA\dataset_images"

def count_png(folder):
    c = 0
    for r, _, files in os.walk(folder):
        c += sum(1 for f in files if f.lower().endswith(".png"))
    return c

for split in ["train","val","test"]:
    print(split, count_png(os.path.join(root, split)))

print("TOTAL:", count_png(root))

train 10880
val 2320
test 2360
TOTAL: 15560


In [27]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = df[df["split"]=="train"]["Label"]

classes = np.unique(labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=labels
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{'AD': 2.412280701754386, 'CN': 1.1752136752136753, 'MCI': 0.5765199161425576}


In [28]:
print(df.groupby(["split","Label"]).size())

split  Label
test   AD        11
       CN        22
       MCI       26
train  AD        38
       CN        78
       MCI      159
val    AD        13
       CN        20
       MCI       27
dtype: int64
